In [1]:
from os import cpu_count

import torch
from pylate import models, indexes

/Users/anjali/PycharmProjects/search_modernColBERT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [4]:
# Load the ColBERT model
model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device = device
)

In [5]:
# Initialize the PLAID Index
index = indexes.PLAID(
    index_folder = "../data/pylate_index",
    index_name = "esci_data_index",
    override = True
)

In [6]:
# Data streaming
import sqlite3

conn = sqlite3.connect("../data/processed/products_dataset.db")
cursor = conn.cursor()
cursor.execute("SELECT pid, product_text FROM products LIMIT 1000")

In [ ]:
# Batching and encoding
import gc

batch_texts, batch_ids = [], []
BATCH_SIZE = 32

for i, (pid, text) in enumerate(cursor):
    batch_texts.append(text)
    batch_ids.append(pid)

    if len(batch_texts) == BATCH_SIZE:
        embeddings = model.encode(batch_texts, is_query=False, batch_size=BATCH_SIZE)
        index.add_documents(documents_ids=batch_ids, documents_embeddings=embeddings)

        # Clearing memory
        batch_texts, batch_ids = [], []
        if device == "mps":
            torch.mps.empty_cache()
        gc.collect()

        if i % 1000 == 0:
            print(f"Processed {i} items")

if batch_texts:
    embeddings = model.encode(batch_texts, is_query=False)
    index.add_documents(documents_ids=batch_ids, documents_embeddings=embeddings)

conn.close()